# FINAL PROJECT - BIG DATA DAN DATA MINING
## Analisis dan Prediksi Data dengan Pendekatan Big Data Mining

---

**Metode yang digunakan:**
1. **Klasifikasi** (Random Forest, Decision Tree, KNN) - Predictive Analytics
2. **Clustering** (K-Means, Hierarchical) - Descriptive Analytics
3. **Association Rule Mining** (Apriori) - Descriptive Analytics

**Dataset Publik:**
- **Telco Customer Churn** (IBM) - untuk Klasifikasi & Clustering
- **Groceries Dataset** (UCI/R) - untuk Association Rule Mining

---

## BAGIAN 1: IMPORT LIBRARY

In [ ]:
# ============================================================================
# BAGIAN 1: IMPORT LIBRARY
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import ssl
import urllib.request
warnings.filterwarnings('ignore')

# Fix SSL certificate issue untuk macOS
ssl._create_default_https_context = ssl._create_unverified_context

# Library untuk Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, silhouette_score)

# Library untuk Klasifikasi
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Library untuk Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering

# Library untuk Association Rule Mining
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print("="*70)
print("FINAL PROJECT - BIG DATA DAN DATA MINING")
print("="*70)
print("\nSemua library berhasil diimport!")

---
## BAGIAN 2: LOAD DATASET PUBLIK DARI URL

### Dataset yang digunakan:
1. **Telco Customer Churn** 
   - Sumber: IBM GitHub
   - URL: https://github.com/IBM/telco-customer-churn-on-icp4d
   - Kegunaan: Klasifikasi & Clustering

2. **Groceries Dataset**
   - Sumber: Machine Learning with R Datasets
   - URL: https://github.com/stedy/Machine-Learning-with-R-datasets
   - Kegunaan: Association Rule Mining

In [2]:
# ============================================================================
# BAGIAN 2: LOAD DATASET DARI URL
# ============================================================================
print("\n" + "="*70)
print("BAGIAN 2: LOAD DATASET PUBLIK")
print("="*70)

# 2.1 Load Dataset Telco Customer Churn
print("\n[2.1] Loading Telco Customer Churn Dataset...")
print("-"*50)

URL_TELCO = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df_churn = pd.read_csv(URL_TELCO)

print(f"URL: {URL_TELCO}")
print(f"\nDataset berhasil dimuat!")
print(f"Ukuran: {df_churn.shape[0]} baris x {df_churn.shape[1]} kolom")


BAGIAN 2: LOAD DATASET PUBLIK

[2.1] Loading Telco Customer Churn Dataset...
--------------------------------------------------


NameError: name 'pd' is not defined

In [ ]:
# 2.2 Load Dataset Groceries untuk Association Rule Mining
print("\n[2.2] Loading Groceries Dataset...")
print("-"*50)

URL_GROCERIES = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv"

# Baca sebagai transaksi (setiap baris adalah satu transaksi)
transactions = []
with urllib.request.urlopen(URL_GROCERIES) as response:
    content = response.read().decode('utf-8')
    for line in content.strip().split('\n'):
        items = [item.strip() for item in line.split(',') if item.strip()]
        if items:
            transactions.append(items)

print(f"URL: {URL_GROCERIES}")
print(f"\nDataset berhasil dimuat!")
print(f"Jumlah transaksi: {len(transactions)}")

# Hitung produk unik
all_items = set()
for trans in transactions:
    all_items.update(trans)
print(f"Jumlah produk unik: {len(all_items)}")

In [ ]:
# 2.3 Preview Dataset Telco Churn
print("\n[2.3] Preview Dataset Telco Customer Churn")
print("-"*50)
print(f"\nKolom dataset: {list(df_churn.columns)}")
df_churn.head(10)

In [ ]:
# 2.4 Preview Transaksi Groceries
print("\n[2.4] Preview Dataset Groceries (Transaksi)")
print("-"*50)
print("\n5 Transaksi pertama:")
for i, trans in enumerate(transactions[:5], 1):
    print(f"  Transaksi {i}: {trans}")

---
## BAGIAN 3: EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# ============================================================================
# BAGIAN 3: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================================
print("\n" + "="*70)
print("BAGIAN 3: EXPLORATORY DATA ANALYSIS (EDA)")
print("="*70)

# 3.1 Info Dataset
print("\n[3.1] Informasi Dataset Telco Churn")
print("-"*50)
df_churn.info()

In [ ]:
# 3.2 Statistik Deskriptif
print("\n[3.2] Statistik Deskriptif (Numerik)")
print("-"*50)
df_churn.describe()

In [ ]:
# 3.3 Distribusi Target (Churn)
print("\n[3.3] Distribusi Target Variable (Churn)")
print("-"*50)

churn_counts = df_churn['Churn'].value_counts()
print(f"Tidak Churn (No) : {churn_counts['No']} ({churn_counts['No']/len(df_churn)*100:.2f}%)")
print(f"Churn (Yes)      : {churn_counts['Yes']} ({churn_counts['Yes']/len(df_churn)*100:.2f}%)")

# Visualisasi
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['Tidak Churn (No)', 'Churn (Yes)'], 
              [churn_counts['No'], churn_counts['Yes']], 
              color=colors, edgecolor='black')
ax.set_title('Distribusi Customer Churn', fontsize=14, fontweight='bold')
ax.set_ylabel('Jumlah Customer')
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}\n({height/len(df_churn)*100:.1f}%)',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Missing Values
print("\n[3.4] Pengecekan Missing Values")
print("-"*50)

missing = df_churn.isnull().sum()
if missing.sum() > 0:
    print("Kolom dengan missing values:")
    print(missing[missing > 0])
else:
    print("Tidak ada missing values (NULL)!")

# Cek nilai kosong string di TotalCharges
empty_total = (df_churn['TotalCharges'] == ' ').sum()
print(f"\nNilai kosong (string) di TotalCharges: {empty_total}")

In [ ]:
# 3.5 Distribusi Fitur Numerik
print("\n[3.5] Distribusi Fitur Numerik")
print("-"*50)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Tenure
axes[0].hist(df_churn['tenure'], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribusi Tenure', fontweight='bold')
axes[0].set_xlabel('Tenure (bulan)')
axes[0].set_ylabel('Frekuensi')

# MonthlyCharges
axes[1].hist(df_churn['MonthlyCharges'], bins=30, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribusi Monthly Charges', fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')

# SeniorCitizen
senior_counts = df_churn['SeniorCitizen'].value_counts()
axes[2].bar(['Non-Senior (0)', 'Senior (1)'], senior_counts.values, 
            color=['#2ecc71', '#9b59b6'], edgecolor='black')
axes[2].set_title('Distribusi Senior Citizen', fontweight='bold')
axes[2].set_ylabel('Jumlah')

plt.tight_layout()
plt.show()

---
## BAGIAN 4: DATA PREPROCESSING

In [ ]:
# ============================================================================
# BAGIAN 4: DATA PREPROCESSING
# ============================================================================
print("\n" + "="*70)
print("BAGIAN 4: DATA PREPROCESSING")
print("="*70)

# 4.1 Copy dataset
print("\n[4.1] Preprocessing Dataset")
print("-"*50)

df = df_churn.copy()

# Hapus customerID
df = df.drop('customerID', axis=1)
print("- Kolom 'customerID' dihapus (tidak relevan untuk analisis)")

# Konversi TotalCharges ke numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("- Kolom 'TotalCharges' dikonversi ke numeric")

# Isi missing dengan median
median_total = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_total, inplace=True)
print(f"- Missing values di 'TotalCharges' diisi dengan median ({median_total:.2f})")

print(f"\nUkuran dataset: {df.shape[0]} baris x {df.shape[1]} kolom")

In [ ]:
# 4.2 Encoding Categorical Variables
print("\n[4.2] Encoding Categorical Variables")
print("-"*50)

# Identifikasi kolom kategorikal
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Kolom kategorikal ({len(cat_cols)}): {cat_cols}")

# Label Encoding untuk target
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])  # No=0, Yes=1
print("\n- Target 'Churn' di-encode: No=0, Yes=1")

# One-Hot Encoding untuk fitur kategorikal
cat_cols.remove('Churn')
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"- One-Hot Encoding diterapkan pada {len(cat_cols)} kolom")
print(f"\nUkuran setelah encoding: {df_encoded.shape[0]} baris x {df_encoded.shape[1]} kolom")

In [ ]:
# 4.3 Split Features dan Target
print("\n[4.3] Split Features dan Target")
print("-"*50)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Jumlah fitur (X): {X.shape[1]}")
print(f"Jumlah sampel: {X.shape[0]}")
print(f"\nDistribusi target (y):")
print(f"  Tidak Churn (0): {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"  Churn (1)      : {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")

In [ ]:
# 4.4 Normalisasi
print("\n[4.4] Normalisasi Data (StandardScaler)")
print("-"*50)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data berhasil dinormalisasi!")
print(f"Mean: {X_scaled.mean():.6f}")
print(f"Std : {X_scaled.std():.6f}")

In [ ]:
# 4.5 Split Training dan Testing
print("\n[4.5] Split Data Training dan Testing")
print("-"*50)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data Training: {len(X_train)} sampel (80%)")
print(f"Data Testing : {len(X_test)} sampel (20%)")

---
## BAGIAN 5: IMPLEMENTASI KLASIFIKASI

Model yang digunakan:
1. Random Forest Classifier
2. Decision Tree Classifier
3. K-Nearest Neighbors (KNN)

In [ ]:
# ============================================================================
# BAGIAN 5: IMPLEMENTASI KLASIFIKASI
# ============================================================================
print("\n" + "="*70)
print("BAGIAN 5: IMPLEMENTASI KLASIFIKASI")
print("="*70)

# Dictionary untuk menyimpan hasil
hasil_klasifikasi = {}

# 5.1 Random Forest
print("\n[5.1] Random Forest Classifier")
print("-"*50)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
hasil_klasifikasi['Random Forest'] = rf_acc

print(f"Akurasi: {rf_acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=['Tidak Churn', 'Churn']))

In [ ]:
# 5.2 Decision Tree
print("\n[5.2] Decision Tree Classifier")
print("-"*50)

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
hasil_klasifikasi['Decision Tree'] = dt_acc

print(f"Akurasi: {dt_acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, dt_pred, target_names=['Tidak Churn', 'Churn']))

In [ ]:
# 5.3 K-Nearest Neighbors
print("\n[5.3] K-Nearest Neighbors (KNN)")
print("-"*50)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)
hasil_klasifikasi['KNN'] = knn_acc

print(f"Akurasi: {knn_acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, knn_pred, target_names=['Tidak Churn', 'Churn']))

In [ ]:
# 5.4 Cross-Validation
print("\n[5.4] Cross-Validation (5-Fold)")
print("-"*50)

models = {
    'Random Forest': rf_model,
    'Decision Tree': dt_model,
    'KNN': knn_model
}

print(f"{'Model':<20} {'Mean CV':>10} {'Std CV':>10}")
print("-"*42)
for name, model in models.items():
    cv_scores = cross_val_score(model, X_scaled, y, cv=5)
    print(f"{name:<20} {cv_scores.mean()*100:>9.2f}% {cv_scores.std()*100:>9.2f}%")

In [ ]:
# 5.5 Perbandingan Model
print("\n[5.5] Perbandingan Akurasi Model")
print("-"*50)

print(f"{'Model':<20} {'Akurasi':>10}")
print("-"*32)
for model, acc in sorted(hasil_klasifikasi.items(), key=lambda x: x[1], reverse=True):
    print(f"{model:<20} {acc*100:>9.2f}%")

best_model = max(hasil_klasifikasi, key=hasil_klasifikasi.get)
print(f"\nModel Terbaik: {best_model} ({hasil_klasifikasi[best_model]*100:.2f}%)")

In [ ]:
# 5.6 Visualisasi Perbandingan & Confusion Matrix
print("\n[5.6] Visualisasi Hasil Klasifikasi")
print("-"*50)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Perbandingan Akurasi
ax1 = axes[0, 0]
models_list = list(hasil_klasifikasi.keys())
accs = [v * 100 for v in hasil_klasifikasi.values()]
colors_bar = ['#3498db', '#9b59b6', '#e67e22']
bars = ax1.bar(models_list, accs, color=colors_bar, edgecolor='black')
ax1.set_title('Perbandingan Akurasi Model Klasifikasi', fontsize=12, fontweight='bold')
ax1.set_ylabel('Akurasi (%)')
ax1.set_ylim(0, 100)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{acc:.1f}%', ha='center', fontweight='bold')

# Plot 2-4: Confusion Matrix
predictions = [rf_pred, dt_pred, knn_pred]
titles = ['Random Forest', 'Decision Tree', 'KNN']
positions = [(0, 1), (1, 0), (1, 1)]

for pred, title, pos in zip(predictions, titles, positions):
    ax = axes[pos]
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Tidak Churn', 'Churn'],
                yticklabels=['Tidak Churn', 'Churn'])
    ax.set_title(f'Confusion Matrix - {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# 5.7 Feature Importance
print("\n[5.7] Feature Importance (Random Forest)")
print("-"*50)

feature_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Fitur Paling Berpengaruh:")
print(feature_imp.head(10).to_string(index=False))

# Visualisasi
fig, ax = plt.subplots(figsize=(10, 6))
top_feat = feature_imp.head(10).sort_values('Importance')
ax.barh(top_feat['Feature'], top_feat['Importance'], color='#3498db', edgecolor='black')
ax.set_xlabel('Importance')
ax.set_title('Top 10 Feature Importance (Random Forest)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## BAGIAN 6: IMPLEMENTASI CLUSTERING

Metode yang digunakan:
1. K-Means Clustering
2. Hierarchical Clustering (Agglomerative)

In [ ]:
# ============================================================================
# BAGIAN 6: IMPLEMENTASI CLUSTERING
# ============================================================================
print("\n" + "="*70)
print("BAGIAN 6: IMPLEMENTASI CLUSTERING")
print("="*70)

# 6.1 Persiapan Data
print("\n[6.1] Persiapan Data untuk Clustering")
print("-"*50)

# Gunakan fitur numerik: tenure, MonthlyCharges, TotalCharges
cluster_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
df_cluster = df[cluster_features].copy()

# Normalisasi
scaler_cluster = StandardScaler()
X_cluster = scaler_cluster.fit_transform(df_cluster)

print(f"Fitur untuk clustering: {cluster_features}")
print(f"Jumlah sampel: {len(X_cluster)}")

In [ ]:
# 6.2 Elbow Method & Silhouette
print("\n[6.2] Menentukan Jumlah Cluster Optimal")
print("-"*50)

inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_cluster, kmeans.labels_))

print(f"{'K':>3} {'Inertia':>12} {'Silhouette':>12}")
print("-"*30)
for k, iner, sil in zip(K_range, inertias, silhouettes):
    print(f"{k:>3} {iner:>12.2f} {sil:>12.4f}")

optimal_k = list(K_range)[np.argmax(silhouettes)]
print(f"\nJumlah cluster optimal (Silhouette tertinggi): K = {optimal_k}")

In [ ]:
# 6.3 Visualisasi Elbow & Silhouette
print("\n[6.3] Visualisasi Elbow Method & Silhouette Score")
print("-"*50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=optimal_k, color='red', linestyle='--', linewidth=2, label=f'Optimal K={optimal_k}')
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Jumlah Cluster (K)')
axes[0].set_ylabel('Inertia (Within-cluster SSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette
axes[1].plot(list(K_range), silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=optimal_k, color='red', linestyle='--', linewidth=2, label=f'Optimal K={optimal_k}')
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Jumlah Cluster (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6.4 K-Means Clustering
print(f"\n[6.4] K-Means Clustering (K={optimal_k})")
print("-"*50)

kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_cluster['Cluster_KMeans'] = kmeans_model.fit_predict(X_cluster)

print("Distribusi Cluster:")
km_counts = df_cluster['Cluster_KMeans'].value_counts().sort_index()
for cluster, count in km_counts.items():
    print(f"  Cluster {cluster}: {count} customers ({count/len(df_cluster)*100:.1f}%)")

sil_kmeans = silhouette_score(X_cluster, df_cluster['Cluster_KMeans'])
print(f"\nSilhouette Score: {sil_kmeans:.4f}")

In [ ]:
# 6.5 Hierarchical Clustering
print(f"\n[6.5] Hierarchical Clustering (K={optimal_k})")
print("-"*50)

hc_model = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
df_cluster['Cluster_HC'] = hc_model.fit_predict(X_cluster)

print("Distribusi Cluster:")
hc_counts = df_cluster['Cluster_HC'].value_counts().sort_index()
for cluster, count in hc_counts.items():
    print(f"  Cluster {cluster}: {count} customers ({count/len(df_cluster)*100:.1f}%)")

sil_hc = silhouette_score(X_cluster, df_cluster['Cluster_HC'])
print(f"\nSilhouette Score: {sil_hc:.4f}")

In [ ]:
# 6.6 Perbandingan Clustering
print("\n[6.6] Perbandingan Metode Clustering")
print("-"*50)

print(f"K-Means Silhouette Score     : {sil_kmeans:.4f}")
print(f"Hierarchical Silhouette Score: {sil_hc:.4f}")

best_clustering = 'K-Means' if sil_kmeans >= sil_hc else 'Hierarchical'
print(f"\nMetode Terbaik: {best_clustering}")

In [ ]:
# 6.7 Karakteristik Cluster
print("\n[6.7] Karakteristik Cluster (K-Means)")
print("-"*50)

cluster_summary = df_cluster.groupby('Cluster_KMeans')[cluster_features].agg(['mean', 'std']).round(2)
print(cluster_summary)

In [ ]:
# 6.8 Visualisasi Cluster
print("\n[6.8] Visualisasi Clustering")
print("-"*50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means
scatter1 = axes[0].scatter(df_cluster['tenure'], df_cluster['MonthlyCharges'],
                           c=df_cluster['Cluster_KMeans'], cmap='viridis', alpha=0.6, s=30)
axes[0].set_title(f'K-Means Clustering (K={optimal_k})\nSilhouette={sil_kmeans:.4f}', fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Monthly Charges ($)')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Hierarchical
scatter2 = axes[1].scatter(df_cluster['tenure'], df_cluster['MonthlyCharges'],
                           c=df_cluster['Cluster_HC'], cmap='viridis', alpha=0.6, s=30)
axes[1].set_title(f'Hierarchical Clustering (K={optimal_k})\nSilhouette={sil_hc:.4f}', fontweight='bold')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Monthly Charges ($)')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

---
## BAGIAN 7: ASSOCIATION RULE MINING

Dataset: Groceries (transaksi belanja)
Algoritma: Apriori

In [ ]:
# ============================================================================
# BAGIAN 7: ASSOCIATION RULE MINING
# ============================================================================
print("\n" + "="*70)
print("BAGIAN 7: ASSOCIATION RULE MINING")
print("="*70)

# 7.1 Encoding Transaksi
print("\n[7.1] Encoding Transaksi")
print("-"*50)

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_array, columns=te.columns_)

# Konversi nama kolom ke string
df_trans.columns = [str(col) for col in df_trans.columns]

print(f"Jumlah transaksi: {len(transactions)}")
print(f"Jumlah produk unik: {len(te.columns_)}")
print(f"\n10 Produk pertama: {list(df_trans.columns)[:10]}")

In [ ]:
# 7.2 Frequent Itemsets
print("\n[7.2] Frequent Itemsets (Apriori)")
print("-"*50)

frequent_itemsets = apriori(df_trans, min_support=0.01, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print(f"Jumlah frequent itemsets: {len(frequent_itemsets)}")
print(f"\nTop 15 Frequent Itemsets:")
print(f"{'Itemset':<45} {'Support':>10}")
print("-"*57)

top_items = frequent_itemsets.nlargest(15, 'support')
for _, row in top_items.iterrows():
    items_str = ', '.join([str(i) for i in row['itemsets']])
    print(f"{items_str:<45} {row['support']:>10.4f}")

In [ ]:
# 7.3 Association Rules
print("\n[7.3] Association Rules")
print("-"*50)

# Konversi itemsets
frequent_itemsets['itemsets'] = frequent_itemsets['itemsets'].apply(
    lambda x: frozenset([str(item) for item in x])
)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0,
                          num_itemsets=len(frequent_itemsets))

if len(rules) > 0:
    rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join([str(i) for i in x]))
    rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join([str(i) for i in x]))
    
    print(f"Jumlah rules ditemukan: {len(rules)}")
    print(f"\nTop 15 Rules (berdasarkan Lift):")
    print(f"{'Antecedent':<25} {'Consequent':<20} {'Supp':>7} {'Conf':>7} {'Lift':>7}")
    print("-"*72)
    
    top_rules = rules.nlargest(15, 'lift')
    for _, row in top_rules.iterrows():
        ant = row['antecedents_str'][:23]
        cons = row['consequents_str'][:18]
        print(f"{ant:<25} {cons:<20} {row['support']:>7.4f} {row['confidence']:>7.4f} {row['lift']:>7.4f}")
else:
    print("Tidak ada rules yang memenuhi kriteria.")

In [ ]:
# 7.4 Interpretasi Rules
print("\n[7.4] Interpretasi Association Rules")
print("-"*50)

if len(rules) > 0:
    best_rule = rules.loc[rules['lift'].idxmax()]
    
    print("Rule dengan Lift Tertinggi:")
    print(f"\n  IF customer membeli: [{best_rule['antecedents_str']}]")
    print(f"  THEN kemungkinan juga membeli: [{best_rule['consequents_str']}]")
    print(f"\n  Metrics:")
    print(f"    Support   : {best_rule['support']:.4f} ({best_rule['support']*100:.2f}% transaksi)")
    print(f"    Confidence: {best_rule['confidence']:.4f} ({best_rule['confidence']*100:.2f}%)")
    print(f"    Lift      : {best_rule['lift']:.4f}")
    
    print(f"\n  Interpretasi:")
    print(f"    Lift > 1 menunjukkan adanya asosiasi positif antara produk.")
    print(f"    Customer yang membeli {best_rule['antecedents_str']} memiliki")
    print(f"    kemungkinan {best_rule['lift']:.2f}x lebih tinggi untuk membeli {best_rule['consequents_str']}.")

In [ ]:
# 7.5 Visualisasi Association Rules
print("\n[7.5] Visualisasi Association Rules")
print("-"*50)

if len(rules) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    scatter = ax.scatter(rules['support'], rules['confidence'],
                        c=rules['lift'], cmap='viridis', alpha=0.6, s=50, edgecolor='black')
    
    ax.set_xlabel('Support', fontsize=11)
    ax.set_ylabel('Confidence', fontsize=11)
    ax.set_title('Association Rules\n(Support vs Confidence, colored by Lift)', fontsize=12, fontweight='bold')
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Lift', fontsize=11)
    
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada rules untuk divisualisasikan.")

---
## BAGIAN 8: KESIMPULAN DAN REKOMENDASI

In [ ]:
# ============================================================================
# BAGIAN 8: KESIMPULAN DAN REKOMENDASI
# ============================================================================
print("\n" + "="*70)
print("BAGIAN 8: KESIMPULAN DAN REKOMENDASI")
print("="*70)

print("\n" + "="*70)
print("RINGKASAN HASIL ANALISIS")
print("="*70)

print("\n[1] KLASIFIKASI - Customer Churn Prediction")
print("-"*50)
print(f"    Dataset      : Telco Customer Churn (IBM)")
print(f"    Jumlah Data  : {len(df_churn)} customers")
print(f"    Model Terbaik: {best_model}")
print(f"    Akurasi      : {hasil_klasifikasi[best_model]*100:.2f}%")
print(f"    Top Feature  : {feature_imp.iloc[0]['Feature']}")

print("\n[2] CLUSTERING - Customer Segmentation")
print("-"*50)
print(f"    Dataset       : Telco Customer Churn (IBM)")
print(f"    Jumlah Cluster: {optimal_k}")
print(f"    Metode Terbaik: {best_clustering}")
print(f"    Silhouette    : {max(sil_kmeans, sil_hc):.4f}")

print("\n[3] ASSOCIATION RULE MINING - Market Basket Analysis")
print("-"*50)
print(f"    Dataset          : Groceries")
print(f"    Jumlah Transaksi : {len(transactions)}")
print(f"    Frequent Itemsets: {len(frequent_itemsets)}")
print(f"    Association Rules: {len(rules)}")

In [ ]:
print("\n" + "="*70)
print("REKOMENDASI BISNIS")
print("="*70)

print("""
1. CHURN PREVENTION (Berdasarkan Klasifikasi)
   - Fokus pada customer dengan tenure rendah (pelanggan baru)
   - Monitor customer dengan kontrak month-to-month (rentan churn)
   - Berikan program loyalitas untuk meningkatkan retention
   - Perhatikan customer dengan monthly charges tinggi

2. CUSTOMER SEGMENTATION (Berdasarkan Clustering)
   - Segmentasi customer berdasarkan tenure dan monthly charges
   - Strategi marketing berbeda untuk setiap segmen
   - Identifikasi high-value customers untuk program VIP
   - Tawarkan upgrade layanan ke segmen yang sesuai

3. PRODUCT BUNDLING (Berdasarkan Association Rules)
   - Buat bundle produk berdasarkan pola pembelian
   - Implementasi cross-selling recommendation
   - Optimasi product placement di toko
   - Promo bundle untuk produk dengan lift tinggi
""")

print("="*70)
print("PROGRAM FINAL PROJECT SELESAI")
print("="*70)